In [1]:
!pip install fastapi uvicorn supabase httpx pydantic edge-tts nest-asyncio python-dotenv

In [ ]:
import os
import asyncio
import json
import re
import gc
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from supabase import create_client, Client
import nest_asyncio
import uvicorn
import httpx
import edge_tts
from fastapi.responses import Response

nest_asyncio.apply()

app = FastAPI(title="Stateful AI Voice Automator v4", version="4.0.0")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

try:
    supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)
    print("Connected securely to Supabase!")
except Exception as e:
    print(f"Supabase Init Failed: {e}")

SESSION_MEMORY = {}

def fetch_live_menu() -> str:
    try:
        response = supabase.table("menu_items").select("name, price").eq("is_available", True).execute()
        return ", ".join([f"{item['name']} (${item['price']})" for item in response.data])
    except Exception:
        return "Double Cheeseburger ($9.99), Classic Hamburger ($7.49), French Fries ($3.49), Coca Cola ($1.99)"

async def extract_and_commit_order(session_id: str, history_list: list):
    """Analyzes the chat log independently using a high-speed pass to log data to Supabase."""
    try:
        print(f"Background extractor running for session {session_id}...")
        formatted_history = "\n".join([f"{m['role'].upper()}: {m['content']}" for m in history_list])

        extraction_prompt = (
            "Analyze the following restaurant order dialogue and extract the finalized details into a clean JSON object.\n"
            "Return ONLY a raw JSON block containing keys: customer_name, phone_number, delivery_address, items (an array of objects with 'name' and 'quantity').\n"
            "Do not include any chat commentary or markdown backticks.\n\n"
            f"Dialogue Transcript:\n{formatted_history}"
        )

        async with httpx.AsyncClient(verify=False) as client:
            headers = {"Authorization": f"Bearer {GROQ_API_KEY}", "Content-Type": "application/json"}
            payload = {
                "model": "llama-3.1-8b-instant",
                "messages": [{"role": "user", "content": extraction_prompt}],
                "temperature": 0.0,
                "max_tokens": 300
            }
            res = await client.post("https://api.groq.com/openai/v1/chat/completions", headers=headers, json=payload, timeout=10.0)
            raw_json_text = res.json()["choices"][0]["message"]["content"].strip()

        raw_json_text = raw_json_text.replace("```json", "").replace("```", "").strip()
        structured_data = json.loads(raw_json_text)

        phone = structured_data.get("phone_number", "+i0000000000")
        name = structured_data.get("customer_name", "Voice Customer")

        customer_query = supabase.table("customers").select("id").eq("phone_number", phone).execute()
        customer_id = customer_query.data[0]["id"] if customer_query.data else supabase.table("customers").insert({
            "phone_number": phone, "name": name, "password_hash": "$2b$12$VoiceGeneratedDummyHash",
            "default_delivery_address": structured_data.get("delivery_address", "Pickup")
        }).execute().data[0]["id"]

        supabase.table("sessions").update({"customer_id": customer_id}).eq("id", session_id).execute()

        ordered_items = structured_data.get("items", [])
        calculated_total = 0.00
        verified_items = []

        for o_item in ordered_items:
            item_name = o_item.get("name", "")
            qty = int(o_item.get("quantity", 1))
            menu_lookup = supabase.table("menu_items").select("id", "price").ilike("name", item_name).execute()
            if menu_lookup.data:
                calculated_total += float(menu_lookup.data[0]["price"]) * qty
                verified_items.append({"id": menu_lookup.data[0]["id"], "price": menu_lookup.data[0]["price"], "qty": qty})

        order_insert = supabase.table("orders").insert({
            "customer_id": customer_id, "session_id": session_id,
            "total_amount": calculated_total, "status": "pending",
            "delivery_address": structured_data.get("delivery_address", "Address Provided on Voice")
        }).execute()
        order_id = order_insert.data[0]["id"]

        for v_item in verified_items:
            supabase.table("order_items").insert({
                "order_id": order_id, "menu_item_id": v_item["id"],
                "quantity": v_item["qty"], "price_at_purchase": v_item["price"]
            }).execute()

        print(f"Database successfully locked for Order #{order_id} (Total: ${calculated_total})!")
        return True
    except Exception as e:
        print(f"Background data parsing broke: {e}")
        return False


@app.post("/process-audio")
async def process_audio(
    file: UploadFile = File(...),
    session_id: str = Form(...),
    customer_id: str = Form(None)
):
    try:
        audio_bytes = await file.read()
        if not audio_bytes:
            raise HTTPException(status_code=400, detail="Empty files.")

        # 1. State Retrieval & Session Initialization
        session_query = supabase.table("sessions").select("current_step").eq("id", session_id).execute()
        if not session_query.data:
            insert_data = {"id": session_id, "current_step": "welcome"}
            if customer_id and customer_id != "undefined":
                insert_data["customer_id"] = int(customer_id)
            supabase.table("sessions").insert(insert_data).execute()
            current_step = "welcome"
            SESSION_MEMORY[session_id] = []
        else:
            current_step = session_query.data[0]["current_step"]
            if session_id not in SESSION_MEMORY:
                SESSION_MEMORY[session_id] = []

        # 2. Whisper Speech-to-Text Transcription Passage
        async with httpx.AsyncClient() as client:
            headers = {"Authorization": f"Bearer {GROQ_API_KEY}"}
            stt_response = await client.post(
                "https://api.groq.com/openai/v1/audio/transcriptions",
                headers=headers, files={"file": ("audio.webm", audio_bytes, "audio/webm")},
                data={"model": "whisper-large-v3"}, timeout=10.0
            )
            user_text = stt_response.json().get("text", "").strip()
            print(f"[Session: {session_id}] User said: '{user_text}'")

        if not user_text or len(user_text) < 2:
            user_text = "[Garbled sound or unclear response]"

        # 3. CRITICAL: Identity Ingestion Matrix
        customer_profile_context = "Customer Status: Unknown. Ask for name, phone, and address to sign them up."
        active_customer_id = customer_id if (customer_id and customer_id != "undefined") else None

        # Fallback to phone regex lookup if guest user speaks their phone number during the call
        if not active_customer_id:
            all_text = " ".join([m['content'] for m in SESSION_MEMORY[session_id]]) + " " + user_text
            phone_match = re.search(r'\+?\d{10,15}', all_text)
            if phone_match:
                detected_phone = phone_match.group(0)
                customer_check = supabase.table("customers").select("id").eq("phone_number", detected_phone).execute()
                if customer_check.data:
                    active_customer_id = customer_check.data[0]["id"]

        # If user is verified via Login OR spoken phone digits, inject profile data parameters
        if active_customer_id:
            customer_query = supabase.table("customers").select("*").eq("id", int(active_customer_id)).execute()
            if customer_query.data:
                profile = customer_query.data[0]
                customer_profile_context = (
                    f"Customer Status: AUTHENTICATED ALREADY LOGGED IN.\n"
                    f"- Name: {profile['name']}\n"
                    f"- Phone Number: {profile['phone_number']}\n"
                    f"- Default Saved Address: {profile['default_delivery_address']}\n"
                    f"CRITICAL DIRECTIVES:\n"
                    f"1. NEVER ask for their name or phone number.\n"
                    f"2. You MUST read out and confirm their 'Default Saved Address'.\n"
                    f"3. If they ask to update or change the address to a new location, capture the new address string and append [UPDATE_ADDRESS: 'New Address Location String Here'] right next to the step tag."
                )

        live_menu = fetch_live_menu()

        # 4. Strict Sequence Dialogue Prompt Configuration
        system_prompt = (
            "You are an elite voice AI cashier for a premium restaurant. "
            f"Live Menu Database: [{live_menu}].\n\n"
            f"Active Session State: '{current_step}'.\n"
            f"DATABASE CUSTOMER PROFILE LOGS:\n[{customer_profile_context}]\n\n"
            "Follow this conversation sequence strictly:\n"
            "1. 'welcome': Greet them warmly by name if they are logged in, and take their food order.\n"
            "2. 'ordering': Listen to items, check availability, update quantities. Do NOT mention prices yet. "
            "When they are done ordering, explicitly transition to the next step by appending [NEXT_STEP: address].\n"
            "3. 'address': Confirm delivery destination details.\n"
            "   - IF the profile logs say 'AUTHENTICATED ALREADY LOGGED IN', read their saved address back to them politely (e.g., 'Perfect Ahmed, I see your default address is 123 Main St. Should we deliver it there?').\n"
            "   - IF they provide a new location instead, accept it and append [UPDATE_ADDRESS: New Address Location].\n"
            "   - Once address destination parameters are agreed upon, advance immediately by appending [NEXT_STEP: confirming].\n"
            "4. 'confirming': State the complete itemized order list, their total cost, and delivery details. Ask: 'Is this correct?' "
            "The moment they say yes, instantly append [NEXT_STEP: completed].\n"
            "5. 'completed': Say: 'Thank you! Your order has been placed successfully. Have a great day!' and stop talking.\n\n"
            "CRITICAL RULES:\n"
            "- Keep your responses under 2 sentences and natural.\n"
            "- Output ONLY your spoken response followed by the uppercase state marker tags (e.g., [NEXT_STEP: confirming])."
        )

        messages = [{"role": "system", "content": system_prompt}]
        messages.extend(SESSION_MEMORY[session_id][-10:])
        messages.append({"role": "user", "content": user_text})

        # 5. Groq Llama-3 Inference Execution
        async with httpx.AsyncClient() as client:
            headers = {"Authorization": f"Bearer {GROQ_API_KEY}", "Content-Type": "application/json"}
            llm_response = await client.post(
                "https://api.groq.com/openai/v1/chat/completions",
                headers=headers,
                json={"model": "llama-3.1-8b-instant", "messages": messages, "temperature": 0.1, "max_tokens": 150},
                timeout=10.0
            )
            raw_ai_out = llm_response.json()["choices"][0]["message"]["content"].strip()

        # Parse tags safely
        next_step = current_step
        if "[NEXT_STEP:" in raw_ai_out:
            try:
                parts = raw_ai_out.split("[NEXT_STEP:")
                ai_text_response = parts[0].strip()
                parsed_step = parts[1].split("]")[0].strip().lower()
                if parsed_step in ['welcome', 'ordering', 'address', 'confirming', 'completed']:
                    next_step = parsed_step
            except Exception:
                ai_text_response = raw_ai_out
        else:
            ai_text_response = raw_ai_out

        # INTERACTIVE INTERCEPT MATRIX: If user requests a new location address update
        if "[UPDATE_ADDRESS:" in raw_ai_out and active_customer_id:
            try:
                new_address_extracted = raw_ai_out.split("[UPDATE_ADDRESS:")[1].split("]")[0].replace("'", "").strip()
                print(f"🔄 Updating default delivery address for Customer #{active_customer_id} to: '{new_address_extracted}'")
                supabase.table("customers").update({"default_delivery_address": new_address_extracted}).eq("id", int(active_customer_id)).execute()
            except Exception as addr_err:
                print(f"Failed to execute inline address update query: {addr_err}")

        # State transition persistence routing
        if next_step != current_step:
            supabase.table("sessions").update({"current_step": next_step}).eq("id", session_id).execute()
            print(f"State transitioned: {current_step.upper()} to {next_step.upper()}")

            if next_step == "completed":
                print("User confirmed order details. Executing final database commit...")
                asyncio.create_task(extract_and_commit_order(session_id, SESSION_MEMORY[session_id]))

        clean_speech_text = re.sub(r'\[.*?\]', '', ai_text_response).strip()
        print(f"[Session: {session_id}] AI Response: '{clean_speech_text}'")

        if current_step != "confirming" or next_step != "confirming":
            SESSION_MEMORY[session_id].append({"role": "user", "content": user_text})
            SESSION_MEMORY[session_id].append({"role": "assistant", "content": clean_speech_text})

        # 6. Edge-TTS Audio Generation
        communicate = edge_tts.Communicate(clean_speech_text, "en-IN-PrabhatNeural", rate="+15%")
        output_audio_bytes = b""
        async for chunk in communicate.stream():
            if chunk["type"] == "audio":
                output_audio_bytes += chunk["data"]

        response_obj = Response(content=output_audio_bytes, media_type="audio/mpeg")
        del output_audio_bytes
        gc.collect()
        return response_obj

    except Exception as e:
        print(f"🚨 Pipeline failure: {e}")
        raise HTTPException(status_code=500, detail=str(e))


if __name__ == "__main__":
    config = uvicorn.Config(app=app, host="0.0.0.0", port=8000, loop="asyncio")
    server = uvicorn.Server(config)
    loop = asyncio.get_event_loop()
    loop.create_task(server.serve())
    print("Fixed Production-Ready Voice Backend active on port 8000!")

Connected securely to Supabase!
Fixed Production-Ready Voice Backend active on port 8000!


In [3]:
!pip install pyngrok

In [4]:
from pyngrok import ngrok
import os

# Set your Ngrok Auth Token
NGROK_TOKEN = "3E1DL41X1DCSUTDQQGb61e3dYyH_83FQJgPTbJycpRHoaTnK9"
ngrok.set_auth_token(NGROK_TOKEN)

# Open an HTTP tunnel on port 8000
public_url = ngrok.connect(8000)
print(f"🚀 Your secure Public Frontend Bridge URL is: {public_url}")

🚀 Your secure Public Frontend Bridge URL is: NgrokTunnel: "https://embassy-specimen-jersey.ngrok-free.dev" -> "http://localhost:8000"
